# 2. Creating and updating metadata

> 📖 **API reference**: every endpoint used below is documented in the live Swagger UI at [http://localhost:5000/docs](http://localhost:5000/docs) — useful for browsing request/response shapes, trying calls interactively, or finding endpoints not covered here.

This notebook walks through how to add, edit, and organize **metadata records** in the registry from Python — registering a new subject, correcting a wrong field on an existing record, grouping data assets into collections, and so on. All of these go through the auto-generated `biodata_registry_api_client` package, which validates every write against the same rules as the API itself, so records stay well-formed and consistent.

## What this notebook is (and isn't) for

This notebook is about **CRUD** — Create, Read, Update, Delete — on individual metadata records. It's the right tool when you know which record you want to touch (or are creating a new one) and you want the changes to be safe and validated.

This notebook is **not** about flexible search or cross-record analysis. Questions like *"give me all assets for project X with wt/wt subjects between these dates"* are far easier to answer against the joined document store, where everything is denormalized and indexed for read. For those, see [`Integrated_Document_Analysis.ipynb`](./Integrated_Document_Analysis.ipynb) — that notebook currently lives on the `docs/integrated-document-analysis` branch; once both PRs merge it'll be alongside this one.

## How the client is organized

Endpoints are split across three handles:

- `CoreApi` — the science data: subjects, data assets, acquisitions, instruments, processes, schemas, quality control.
- `AdminApi` — administrative records: users, organizations, spaces, collections.
- `ViewsApi` — read-only joined views (e.g. `data_asset_view`), useful when you have a single asset id and want everything tied to it in one call.

## What we'll walk through

1. Set up the client.
2. Look up records — needed before you can edit them.
3. Inspect a schema — needed before you can create new records.
4. Create, update, and delete a record (a complete round-trip on a demo subject).
5. Add and remove assets from a collection.

**Prerequisites**: complete [`1_Set_Up_Local_Database.ipynb`](./1_Set_Up_Local_Database.ipynb) first to get the services running and the database populated.

## 1. Set up the client

Point the client at the running API server and instantiate the three API handles.

In [ ]:
import html as _html
import json as _json
from IPython.display import HTML, display

from biodata_registry_api_client import (
    Configuration,
    ApiClient,
    AdminApi,
    CoreApi,
    ViewsApi,
)

configuration = Configuration(host="http://localhost:5000")
api_client = ApiClient(configuration)

admin_api = AdminApi(api_client)
core_api = CoreApi(api_client)
views_api = ViewsApi(api_client)


# Records returned from the registry are richly nested (schemas, subject details, acquisition
# configs, etc.), so the default `print(...)` repr can take over the whole screen. `show` wraps
# the pretty-printed JSON in a fixed-height scrollable container so cell outputs stay readable.
def show(obj, max_height="320px"):
    """Render obj in a scrollable container. Handles pydantic models and lists of them."""
    if hasattr(obj, "model_dump"):
        payload = obj.model_dump(mode="json")
    elif isinstance(obj, list) and obj and hasattr(obj[0], "model_dump"):
        payload = [o.model_dump(mode="json") for o in obj]
    else:
        payload = obj
    if isinstance(payload, (dict, list)):
        text = _json.dumps(payload, indent=2, default=str)
    else:
        text = repr(payload)
    display(HTML(
        f'<pre style="max-height: {max_height}; overflow: auto; '
        f'background: #f7f7f7; padding: 8px; border-left: 3px solid #ccc; '
        f'font-size: 12px; line-height: 1.4; margin: 4px 0;">'
        f'{_html.escape(text)}</pre>'
    ))


print("client ready")

## 2. Look up a single record by id

The simplest read: when you already know the id of the record you want, hand it to the matching `get_*` endpoint. This is how you'll typically pull a record before updating it.

In [ ]:
user = admin_api.get_user(id=1)
subject = core_api.get_subject(id=1)
data_asset = core_api.get_data_asset(id=1)

show(user)
show(subject)
show(data_asset)

## 3. Pull the joined view of a single asset

If you have one asset id and want everything tied to it (subject, instrument, acquisition) in one call, the `data_asset_view` endpoint gives you that. This is the typical "I want to look at this asset's full context before editing or sharing it" case.

> For *searching* across many assets — filtering by project, genotype, date range, etc. — don't loop over `data_asset_view` calls; use the integrated document store instead. See [`Integrated_Document_Analysis.ipynb`](./Integrated_Document_Analysis.ipynb).

In [ ]:
data_asset_view = views_api.get_data_asset_view(data_asset_id=1)
show(data_asset_view)

## 4. List records with pagination

To browse a table — for example, to find the id of a subject you want to update — use the list endpoints with `limit` and `offset`. A reasonable page size is 100; keep paging until you get a short page.

> The list endpoints don't accept filter expressions (no `genotype=wt/wt`, no date ranges). For anything beyond simple browsing, use [`Integrated_Document_Analysis.ipynb`](./Integrated_Document_Analysis.ipynb).

In [ ]:
subjects = core_api.get_subjects(limit=100, offset=0)
print(f"fetched {len(subjects)} subjects")
show(subjects)

## 5. Inspect a schema

Each record type (subject, acquisition, instrument, ...) is governed by a schema that defines what fields are allowed and required. Looking up the schema is useful both as a sanity check and as a prerequisite for creating new records — the schema id is part of the create call.

In [ ]:
schemas = core_api.get_schemas()
subject_schema = next(s for s in schemas if s.name == "subject")
show(subject_schema)

## 6. Create a new subject record

This is the first half of the CRUD round-trip. To add a new record, build a `*Create` object (here `SubjectCreate`) and pass it to the matching `create_*` endpoint. The `data` field carries the schema-validated payload — same shape as `subject.data` on existing records. If `data` is malformed or missing required fields, the server will reject the call rather than store a half-broken record.

To keep this notebook idempotent across reruns, we use an existing subject as a template and the next two sections will update and then delete what we create here.

In [ ]:
from biodata_registry_api_client import SubjectCreate

template = core_api.get_subject(id=1)
template_data = template.data if isinstance(template.data, dict) else template.data.model_dump()
new_data = {**template_data, "subject_id": "demo_subject"}

subject_create = SubjectCreate(
    name="demo_subject",
    schema_id=subject_schema.id,
    space_id=1,
    data=new_data,
)
new_subject = core_api.create_subject(subject_create=subject_create)
show(new_subject)

## 7. Update an existing record

To modify a record, fetch the current version, build a `*Update` object from it, change whatever fields you want, and send it back. The endpoint expects the full updated record, not a partial diff — that way the server can validate everything together against the schema before saving.

Typical use: correcting a wrong field, adding notes, marking a record as reviewed.

In [ ]:
from biodata_registry_api_client import SubjectUpdate

subject_update = SubjectUpdate(**new_subject.model_dump(mode="json"))
subject_update.data["notes"] = "Hello World!"

updated_subject = core_api.update_subject(id=new_subject.id, subject_update=subject_update)
show(updated_subject)

## 8. Delete a record

Delete by id. Use this carefully — there's no undo at this layer. Doing it here keeps the notebook idempotent: re-running won't accumulate demo records.

In [ ]:
delete_response = core_api.delete_subject(id=updated_subject.id)
print(delete_response)

## 9. Group data assets into a collection

Collections are how you bundle related data assets together — for example, all the assets that backed a particular figure or paper. The link is many-to-many: each asset can belong to several collections, and each collection holds many assets.

You can manage the link from either side: `admin_api.put_collection_data_asset(id=<collection>, data_asset_id=<asset>)` or `core_api.put_data_asset_collection(id=<asset>, collection_id=<collection>)`. Both end up writing the same row.

In [ ]:
my_collection = admin_api.get_collection(id=1)
my_data_asset = core_api.get_data_asset(id=2)

print("collection:")
show(my_collection)
print("asset:")
show(my_data_asset)
print("collection contents before:")
show(admin_api.get_collection_data_assets(id=1))
print("asset's collections before:")
show(core_api.get_data_asset_collections(id=2))

In [ ]:
admin_api.put_collection_data_asset(id=1, data_asset_id=2)

print("collection contents after add:")
show(admin_api.get_collection_data_assets(id=1))
print("asset's collections after add:")
show(core_api.get_data_asset_collections(id=2))

In [ ]:
admin_api.remove_collection_data_asset(id=1, data_asset_id=2)

print("collection contents after remove:")
show(admin_api.get_collection_data_assets(id=1))
print("asset's collections after remove:")
show(core_api.get_data_asset_collections(id=2))